# Part 4 — Harmonizing Datasets

*Written for idtrack 0.1.2.*

This part shows how to use idtrack to bring the gene identifiers of several `.h5ad` datasets to one shared set before you integrate them.

You will learn:
- how to harmonize the features of several datasets, from different releases and of different kinds of identifiers;
- how to read the diagnostics (what was converted, what was not, what is ambiguous);
- how to choose between the **union** and the **intersection** of the features;
- how to bring older data (older ids, older symbols, mixed kinds) to a current release.

> **Install:** harmonization needs the `harmonize` extra, which adds `anndata` and `scipy`: `pip install "idtrack[harmonize]"` (or from GitHub: `pip install "idtrack[harmonize] @ git+https://github.com/theislab/idtrack.git@v0.1.2"`). Everything the harmonizer does to your matrices, column by column: [Harmonizing datasets](../harmonization.rst).

## 4.0 — Why harmonization matters

When you merge datasets, you assume that feature `X` in dataset A is the same gene as feature `X` in dataset B. That breaks when:
- the datasets were annotated with different Ensembl releases (ids were retired, merged, split; symbols renamed);
- one dataset uses HGNC symbols and another Ensembl ids;
- some features map to several genes (1→n) or to none (1→0).

idtrack makes these cases explicit, and its conversions can be reproduced with the same graph.

### 4.0.1 Prerequisites

- The human graph (Part 2) in your local repository.
- The `harmonize` extra (above).
- For section 4.3, your own `.h5ad` files (the datasets of the recorded run are not shipped).

In [1]:
# Load notebook utilities (collapsible output magic for tutorials). `_notebook_utils.py` sits next to the notebooks in
# idtrack's repository; a notebook downloaded on its own runs without it and shows the logs as they come.
try:
    get_ipython().run_line_magic('load_ext', '_notebook_utils')
except ModuleNotFoundError:
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def collapse(line, cell):
        get_ipython().run_cell(cell)

In [2]:
# Setup
from __future__ import annotations

import os
from pathlib import Path

import anndata as ad
import pandas as pd
from scipy import sparse

from idtrack import API
from idtrack.harmonize import HarmonizeFeatures

# The local repository (IDTRACK_LOCAL_REPO is a convention of these tutorials; idtrack reads no such variable).
LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)
SNAPSHOT_RELEASE = 115  # the snapshot release of the graph of Part 2

pd.set_option('display.width', 120)
print('Local repository:', LOCAL_REPOSITORY)

Local repository: idtrack_cache


## 4.1 — A small example (start small, then scale)

Three small `AnnData` objects with overlapping genes but different kinds of identifiers:
- `toy_A`: HGNC symbols of today;
- `toy_B`: symbols and an Ensembl gene id of the same gene (TP53 twice), and a name that is no gene;
- `toy_C`: a dataset annotated with Ensembl release 75 (GRCh37): TP53, BRCA1, symbols that HGNC has renamed since (`MARCH1` is now MARCHF1, `SEPT7` SEPTIN7, `C10orf2` TWNK) and a gene id that Ensembl retired after release 75 (`ENSG00000198477`, ZNF280B).

In [3]:
toy_dir = LOCAL_REPOSITORY / 'toy_harmonization_demo'
toy_dir.mkdir(parents=True, exist_ok=True)

features = {
    'toy_A': ['TP53', 'BRCA1', 'BRCA2', 'BRAF', 'KRAS'],
    'toy_B': ['TP53', 'ENSG00000141510', 'BRCA1', 'NOT_A_REAL_GENE'],
    'toy_C': ['TP53', 'BRCA1', 'MARCH1', 'SEPT7', 'C10orf2', 'ENSG00000198477'],
}
data_h5ad_dict = {}
for seed, (name, genes) in enumerate(features.items()):
    X = sparse.random(40 + 10 * seed, len(genes), density=0.2, format='csr', random_state=seed)
    adata = ad.AnnData(X=X, obs=pd.DataFrame(index=[f'{name}_cell{i}' for i in range(X.shape[0])]),
                       var=pd.DataFrame(index=genes))
    path = toy_dir / f'{name}.h5ad'
    adata.write_h5ad(path)
    data_h5ad_dict[name] = str(path)

data_h5ad_dict

{'toy_A': 'idtrack_cache/toy_harmonization_demo/toy_A.h5ad',
 'toy_B': 'idtrack_cache/toy_harmonization_demo/toy_B.h5ad',
 'toy_C': 'idtrack_cache/toy_harmonization_demo/toy_C.h5ad'}

### 4.1.1 Run the harmonizer

`idtrack.harmonize.HarmonizeFeatures` converts every feature of every dataset when it is made. The results are stored as one JSON file per dataset in `project_local_repository` and reused as long as nothing they depend on changed (the graph's content and the rules of conversion).

The arguments:
- `idtrack_local_repository`: the local repository that holds the graph;
- `snapshot_release`: the snapshot release of that graph;
- `to_release`: the release to harmonize *to* (any release of the graph);
- `final_database`: the database whose names are reported beside each gene (HGNC, MGI, VGNC, ...); the features themselves are Ensembl genes;
- `from_release_by_dataset`, `from_database_by_dataset` (optional, per dataset): the release a dataset was annotated with, and the database its feature names are ids of; nothing is inferred on your behalf;
- `api` (optional): an `idtrack.API` whose graph is already loaded.

`toy_C` is given its release, 75: its features are read as they were at release 75 and followed to release 115.

In [4]:
%%collapse Click to show the conversion log
project_out = toy_dir / 'outputs'
project_out.mkdir(parents=True, exist_ok=True)

harmonizer = HarmonizeFeatures(
    project_name='toy_demo',
    data_h5ad_dict=data_h5ad_dict,
    project_local_repository=str(project_out),
    idtrack_local_repository=str(LOCAL_REPOSITORY),
    to_release=SNAPSHOT_RELEASE,
    final_database='HGNC Symbol',
    organism='homo_sapiens',
    snapshot_release=SNAPSHOT_RELEASE,
    from_release_by_dataset={'toy_C': 75},
    verbosity=0,  # 0: quiet; 1 shows progress bars
)

### 4.1.2 What happened to each feature

`conversion_report()` has one row per input feature of every dataset:
- `status`: `converted` (one gene, with a name in the final database), `converted_no_final_id` (one gene, without one), `ambiguous` (the best answer is several genes: a tie), `no_conversion` (in the graph, but no gene at the target release), `not_in_graph` (no identifier of the graph, or a version it does not hold), `missing_value` or `error`;
- `reason`: why, in words (for a converted feature, empty unless something should be known, such as a reading at another release);
- `ensembl_gene`: the gene at the target release; `final_id`: its name in the final database (`final_id_alternatives`: its other names there);
- `own`: the feature names its gene itself (its stable id, or one of its final-database names). When two features of one dataset reach the same gene, this decides which one stands for the gene (`merge="flag"`, the default; 4.1.3).

In [5]:
report = harmonizer.conversion_report()
display(report[['dataset', 'query_id', 'status', 'ensembl_gene', 'final_id', 'own']])
print(report.groupby('dataset')['status'].value_counts().unstack(fill_value=0))

,dataset,query_id,status,ensembl_gene,final_id,own
0,toy_A,TP53,converted,ENSG00000141510.20,TP53,True
1,toy_A,BRCA1,converted,ENSG00000012048.27,BRCA1,True
2,toy_A,BRCA2,converted,ENSG00000139618.19,BRCA2,True
3,toy_A,BRAF,converted,ENSG00000157764.15,BRAF,True
4,toy_A,KRAS,converted,ENSG00000133703.15,KRAS,True
5,toy_B,TP53,converted,ENSG00000141510.20,TP53,True
6,toy_B,ENSG00000141510,converted,ENSG00000141510.20,TP53,True
7,toy_B,BRCA1,converted,ENSG00000012048.27,BRCA1,True
8,toy_B,NOT_A_REAL_GENE,not_in_graph,NaN,NaN,False
9,toy_C,TP53,converted,ENSG00000141510.20,TP53,True


status   converted  not_in_graph
dataset                         
toy_A            5             0
toy_B            3             1
toy_C            6             0


The renamed symbols of `toy_C` reach today's genes and names (MARCHF1, SEPTIN7, TWNK), and the retired id reaches ZNF280B's gene of GRCh38 by a bridge ([How idtrack answers](../concepts.rst)). `NOT_A_REAL_GENE` is reported, not dropped.

### 4.1.3 One merged AnnData: union or intersection

`unify_multiple_anndatas(features=...)` merges the harmonized datasets:
- `features='union'` (default): every feature of any dataset (a dataset without a feature gets zeros for it);
- `features='intersect'`: only the features present in every dataset.

Each converted gene is one feature, named by its Ensembl gene id at the target release; `var['converted_id']` holds its HGNC symbol (missing for a gene without one: the column never holds an Ensembl id). Nothing is dropped silently: features that are not converted keep their input names, and every feature has `idtrack_status` and `idtrack_reason` per dataset. Two features of one dataset on one gene are summed with `merge='sum'` (for counts); by default (`merge='flag'`) the one that names the gene is kept as the gene's feature and the others are flagged `unconverted_merge_partner`. When several name the gene, as `TP53` and `ENSG00000141510` in `toy_B` do, all are flagged and the gene has no feature of that dataset. `keep_unconverted=False` leaves the unconverted features out.

The merge loses no cell metadata: every `obs` column of any dataset is kept as it is (datasets without a column get missing values there; `obs_join='inner'` keeps only the columns all datasets have, and `obs_columns_to_keep` selects among them), and so are layers, `obsm`, `obsp`, shared `uns` entries and every input `var` column (one column where the datasets agree, else one per dataset). The column `handle_anndata` records the dataset of each cell. `raw` is dropped, because its features are not harmonized.

In [6]:
unified_union = harmonizer.unify_multiple_anndatas(features='union')
unified_intersect = harmonizer.unify_multiple_anndatas(features='intersect')

print('Union shape:', unified_union.shape)
print('Intersect shape:', unified_intersect.shape)
display(unified_union.var[['converted_id', 'idtrack_status_toy_A', 'idtrack_status_toy_B', 'idtrack_status_toy_C']])
print('In every dataset:', list(unified_intersect.var['converted_id']))

Union shape: (150, 12)
Intersect shape: (150, 1)


,converted_id,idtrack_status_toy_A,idtrack_status_toy_B,idtrack_status_toy_C
ENSG00000012048.27,BRCA1,converted,converted,converted
ENSG00000107815.10,TWNK,NaN,NaN,converted
ENSG00000122545.22,SEPTIN7,NaN,NaN,converted
ENSG00000133703.15,KRAS,converted,NaN,NaN
ENSG00000139618.19,BRCA2,converted,NaN,NaN
ENSG00000141510,NaN,NaN,unconverted_merge_partner,NaN
ENSG00000141510.20,TP53,converted,NaN,converted
ENSG00000145416.14,MARCHF1,NaN,NaN,converted
ENSG00000157764.15,BRAF,converted,NaN,NaN
ENSG00000275004.4,ZNF280B,NaN,NaN,converted


In every dataset: ['BRCA1']


The intersection holds BRCA1 alone: TP53 has no feature of `toy_B`, because both of its features there name it (`unconverted_merge_partner`); with `merge='sum'` they would be summed into one.

The status vocabularies, side by side: a feature's `idtrack_status` in the merged object is its `conversion_report()` status (`converted`, `converted_no_final_id`) or, for a feature kept under its input name, `unconverted_` and that status (`unconverted_ambiguous`, `unconverted_not_in_graph`, ...), plus `merged` / `merged_no_final_id` (with `merge='sum'`) and `unconverted_merge_partner`. How they map onto the statuses of `API.convert`: [Harmonizing datasets](../harmonization.rst).

At atlas scale, `union` can make a very wide matrix; if you plan memory-heavy models, start with `intersect` or filter genes first.

## 4.2 — Good practice for integration

- **Harmonize into Ensembl gene ids**, the stable backbone; use symbols for reporting (`converted_id`).
- **Treat 1→n answers as real ambiguity**, not as a nuisance. The harmonizer keeps an ambiguous feature under its input name (status `unconverted_ambiguous`, the genes in its reason). Decide a policy:
  - drop them (`keep_unconverted=False` drops every unconverted feature);
  - keep them as they are (the default);
  - resolve them with what the graph cannot know: the release a dataset was annotated with (`from_release_by_dataset`) or the database its names come from (`from_database_by_dataset`).
- **Give each dataset its release** where you know it (the release of its GTF or reference package): a name is then read as it was meant then.
- **Pilot first**: harmonize a subset and read the diagnostics before scaling up.
- **Union or intersection**: `union` keeps more features but makes a wider matrix; `intersect` is stricter and often more comparable, but can drop genes that matter.

> **Expected result:** you can justify, and report, your integration choice.

## 4.3 — At atlas scale: a recorded run, and your own datasets

The logic is the same for large datasets; what you manage in addition is many files, tens of thousands of features per dataset, the target release, and the disk for the result cache (one JSON file per dataset).

### 4.3.1 The recorded run (four datasets of the Human Lung Cell Atlas)

The numbers below come from a run of this section on four datasets of the extended Human Lung Cell Atlas (HLCA; Sikkema et al., Nature Medicine 2023), harmonized to HGNC symbols at release 115 with the human graph of Part 2, without a release per dataset. Two of the four datasets are unpublished, so the data are not shipped, and the cells that made these numbers run only on files you have (4.3.2).

Features per dataset, by `conversion_report()` status:

| dataset | input features | converted | converted_no_final_id | ambiguous | no_conversion | not_in_graph |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Kaminski_2020 | 45,947 | 34,251 | 11,385 | 55 | 256 | 0 |
| Meyer_2021 | 20,922 | 20,485 | 386 | 27 | 24 | 0 |
| MeyerNikolic_unpubl | 33,582 | 25,467 | 7,886 | 38 | 165 | 26 |
| Barbry_unpubl | 16,859 | 15,243 | 1,560 | 13 | 40 | 3 |

Between 99.3 and 99.8 % of the features of each dataset reach one gene of release 115. The `converted_no_final_id` features reach a gene that has no HGNC symbol at release 115; they keep the gene id as feature name and a missing `converted_id`. The merge of the four datasets holds 774,178 cells; `features='union'` gives 50,510 features, `features='intersect'` 13,408 (26.5 % of the union).

### 4.3.2 Run it on your own datasets

Give your files in `MY_DATASETS` (a name per dataset and the path of its `.h5ad` file; the feature names are read from `var_names`) and, where you know them, the releases each dataset was annotated with in `MY_RELEASES`. The cells below then harmonize and merge them; with `MY_DATASETS` empty they do nothing. The merge loads every dataset into memory: run it on a machine with memory for all of them.

In [7]:
# Your datasets: {dataset name: path to its .h5ad file}. Empty: the cells of this section do nothing.
MY_DATASETS: dict[str, str] = {}
# Optional: the Ensembl release each dataset was annotated with, e.g. {'study_2019': 93}.
MY_RELEASES: dict[str, int] = {}

missing = {name: path for name, path in MY_DATASETS.items() if not os.path.isfile(path)}
print('Datasets given:', len(MY_DATASETS), '| missing files:', len(missing))

Datasets given: 0 | missing files: 0


In [8]:
%%collapse Click to show the conversion log
if MY_DATASETS and not missing:
    my_out = LOCAL_REPOSITORY / 'my_harmonization'
    my_out.mkdir(parents=True, exist_ok=True)
    my_harmonizer = HarmonizeFeatures(
        project_name='my_project',
        data_h5ad_dict=MY_DATASETS,
        project_local_repository=str(my_out),
        idtrack_local_repository=str(LOCAL_REPOSITORY),
        to_release=SNAPSHOT_RELEASE,  # any release of the graph
        final_database='HGNC Symbol',
        organism='homo_sapiens',
        snapshot_release=SNAPSHOT_RELEASE,
        from_release_by_dataset=MY_RELEASES,
        verbosity=1,
    )
else:
    my_harmonizer = None
    print('No datasets given (MY_DATASETS is empty): nothing to harmonize.')

In [9]:
# The table of 4.3.1 for your datasets: features per dataset by status, and the share that reached one gene
if my_harmonizer is not None:
    my_report = my_harmonizer.conversion_report()
    table = my_report.groupby('dataset', sort=False)['status'].value_counts().unstack(fill_value=0)
    table.insert(0, 'input features', table.sum(axis=1))
    one_gene = table[[c for c in ('converted', 'converted_no_final_id') if c in table]].sum(axis=1)
    table['one gene (%)'] = (100 * one_gene / table['input features']).round(1)
    display(table)

In [10]:
%%collapse Click to show the merge log
if my_harmonizer is not None:
    my_union = my_harmonizer.unify_multiple_anndatas(features='union')
    my_intersect = my_harmonizer.unify_multiple_anndatas(features='intersect')
    print('cells:', my_union.n_obs, '| union:', my_union.n_vars, 'features | intersect:', my_intersect.n_vars, 'features')
    # Optional: save them (can be large)
    # my_union.write_h5ad(my_out / 'my_union_harmonized.h5ad')

## 4.4 — Old data, current release

Old datasets often mix older Ensembl ids, symbols that have changed, and project-specific aliases. A workflow you can reproduce:

1. Fix the **snapshot release** of your graph and the **target release** (the release you convert to).
2. Convert into Ensembl gene ids at the target release.
3. Read the failure and ambiguity counts.
4. Only then convert into names for presentation (HGNC symbols), if needed.

The graph holds every assembly: GRCh37 (and NCBI36) identifiers convert to the release you choose, on the assembly Ensembl maintained at that release (GRCh38 from release 76; GRCh37 for a target release from 55 to 75).

The cell below does this with the API for a short mixed list.

In [11]:
api = API.build('homo_sapiens', SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))

legacy_ids = [
    'TP53',             # HGNC symbol
    'P53',              # a synonym of TP53
    'ENSG00000141510',  # Ensembl gene id (TP53)
    'ENSG00000139618',  # BRCA2
    'C10orf2',          # an older symbol of TWNK
    'NOT_A_REAL_GENE',  # a clean 1→0
]

results = api.convert_many(legacy_ids, to_release=SNAPSHOT_RELEASE)
print(API.classification_summary(API.classify(results, 'all')))

for r in results:
    print(f"{r.query_id:<16} -> {list(r.targets('all'))} | read as {r.graph_id!r} ({r.match}) at {r.from_release}")


IDTrack conversion summary:
  Total processed: 6
  1→0: 1 (16.7%)
  Read at another release: 0 (0.0%)
  1→1: 5 (83.3%)
    Changed only: 3 (60.0%)
    Alternative targets: 0 (0.0%)
    Rest: 2 (40.0%)
  1→n: 0 (0.0%)
    Changed only: 0 (0.0%)
    Alternative targets: 0 (0.0%)
  Statuses:
    converted: 5
    not_found: 1
TP53             -> ['ENSG00000141510.20'] | read as 'TP53' (exact) at 115
P53              -> ['ENSG00000141510.20'] | read as 'P53' (synonym:exact) at 115
ENSG00000141510  -> ['ENSG00000141510.20'] | read as 'ENSG00000141510' (exact) at 115
ENSG00000139618  -> ['ENSG00000139618.19'] | read as 'ENSG00000139618' (exact) at 115
C10orf2          -> ['ENSG00000107815.10'] | read as 'C10ORF2' (synonym:case) at 115
NOT_A_REAL_GENE  -> [] | read as None (None) at None


"Changed only" counts the answers whose target is not the identifier read: a symbol converted to an Ensembl gene id is "changed", an Ensembl id converted to its own current version is not (Part 3, section 3.3).

## 4.5 — Summary

- Harmonize identifiers before integration; otherwise you risk merging features that are not the same gene.
- Use Ensembl gene ids as the integration backbone; keep symbols (or other external ids) for reporting.
- Give each dataset its release where you know it.
- Treat 1→0 and 1→n answers as diagnostics: read them and decide a policy.
- Make both the `union` and the `intersect` object early: they answer different questions downstream.